# 🧠 Redes Neuronales Artificiales - Implementación desde CERO

## Introducción

Bienvenido al notebook de **Redes Neuronales Artificiales** (Artificial Neural Networks - ANN), donde exploraremos desde sus fundamentos matemáticos hasta su implementación completa usando solo NumPy.

Las redes neuronales son el pilar fundamental del **Deep Learning** y han revolucionado campos como visión por computadora, procesamiento de lenguaje natural, y sistemas de recomendación. A diferencia de los algoritmos lineales como la regresión logística, las redes neuronales pueden aprender **representaciones no lineales complejas** mediante capas ocultas de neuronas interconectadas.

### 🎯 Objetivos de Aprendizaje

Al completar este notebook, serás capaz de:

1. **Comprender** la arquitectura de una red neuronal multicapa (perceptrón multicapa)
2. **Implementar** forward propagation para calcular predicciones
3. **Aplicar** funciones de activación no lineales (sigmoid, ReLU, tanh)
4. **Calcular** gradientes usando backpropagation y la regla de la cadena
5. **Entrenar** una red neuronal usando gradient descent
6. **Visualizar** fronteras de decisión no lineales
7. **Comparar** el poder expresivo de redes neuronales vs modelos lineales

### 📚 Contexto Histórico

El concepto de neurona artificial fue introducido por **McCulloch & Pitts (1943)**, seguido por el **Perceptrón de Rosenblatt (1958)**. Sin embargo, el verdadero avance llegó con el algoritmo de **Backpropagation** popularizado por **Rumelhart, Hinton & Williams (1986)**, que permitió entrenar redes multicapa de manera eficiente.

## Tabla de Contenidos

- [1 - Configuración del Entorno](#1)
- [2 - Teoría de Redes Neuronales](#2)
  - [2.1 - Neurona Artificial](#2.1)
  - [2.2 - Funciones de Activación](#2.2)
  - [2.3 - Arquitectura de Red Multicapa](#2.3)
  - [2.4 - Forward Propagation](#2.4)
  - [2.5 - Backpropagation y Gradientes](#2.5)
- [3 - Implementación desde CERO](#3)
  - [3.1 - Clase RedNeuronal Completa](#3.1)
- [4 - Ejercicios GRADED](#4)
  - [Ejercicio 1 - Forward Propagation](#ex01)
  - [Ejercicio 2 - Calcular Gradientes (Backpropagation)](#ex02)
- [5 - Ejemplo Práctico: Clasificación No Lineal](#5)
  - [5.1 - Dataset Moons](#5.1)
  - [5.2 - Entrenamiento de la Red](#5.2)
  - [5.3 - Visualización de Resultados](#5.3)
  - [5.4 - Comparación con Regresión Logística](#5.4)
- [6 - Resumen y Conceptos Clave](#6)
- [7 - Referencias](#7)

<a name='1'></a>
## 1 - Configuración del Entorno

Importamos las bibliotecas necesarias y configuramos el entorno de trabajo.

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.plot_utils import plot_decision_boundary, plot_learning_curve
    from utils.testing_utils import print_success, print_error, print_info
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

<a name='2'></a>
## 2 - Teoría de Redes Neuronales

<a name='2.1'></a>
### 2.1 - Neurona Artificial

Una **neurona artificial** es la unidad básica de una red neuronal. Está inspirada en las neuronas biológicas y realiza dos operaciones fundamentales:

1. **Combinación lineal** de las entradas:
   $$z = \mathbf{w}^T \mathbf{x} + b = \sum_{i=1}^{n} w_i x_i + b$$

2. **Aplicación de función de activación** no lineal:
   $$a = g(z)$$

donde:
- $\mathbf{x} \in \mathbb{R}^n$ es el vector de entrada
- $\mathbf{w} \in \mathbb{R}^n$ es el vector de pesos
- $b \in \mathbb{R}$ es el sesgo (bias)
- $g(\cdot)$ es la función de activación
- $a$ es la activación o salida de la neurona

<a name='2.2'></a>
### 2.2 - Funciones de Activación

Las funciones de activación introducen **no linealidad** en la red, permitiendo aprender patrones complejos.

#### **1. Sigmoid (Logística)**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Derivada:**
$$\frac{d\sigma}{dz} = \sigma(z)(1 - \sigma(z))$$

- **Rango**: $(0, 1)$
- **Uso**: Capas de salida en clasificación binaria
- **Problema**: Vanishing gradient para valores muy grandes o pequeños

#### **2. Tangente Hiperbólica (tanh)**
$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} = \frac{2}{1 + e^{-2z}} - 1$$

**Derivada:**
$$\frac{d\tanh}{dz} = 1 - \tanh^2(z)$$

- **Rango**: $(-1, 1)$
- **Ventaja**: Centrada en cero, mejor que sigmoid para capas ocultas

#### **3. ReLU (Rectified Linear Unit)**
$$\text{ReLU}(z) = \max(0, z)$$

**Derivada:**
$$\frac{d\text{ReLU}}{dz} = \begin{cases} 1 & \text{si } z > 0 \\ 0 & \text{si } z \leq 0 \end{cases}$$

- **Rango**: $[0, \infty)$
- **Ventajas**: Computacionalmente eficiente, evita vanishing gradient
- **Uso**: Función más popular en deep learning moderno

<a name='2.3'></a>
### 2.3 - Arquitectura de Red Multicapa (MLP)

Un **Perceptrón Multicapa** (Multi-Layer Perceptron) consiste en:

1. **Capa de entrada**: Recibe las features $\mathbf{X}$ (sin activación)
2. **Capas ocultas**: Una o más capas de neuronas con activaciones no lineales
3. **Capa de salida**: Produce la predicción final

**Notación:**
- $L$: Número total de capas (sin contar la entrada)
- $n^{[l]}$: Número de neuronas en la capa $l$
- $\mathbf{W}^{[l]}$: Matriz de pesos de la capa $l$ (dimensión $n^{[l]} \times n^{[l-1]}$)
- $\mathbf{b}^{[l]}$: Vector de sesgos de la capa $l$ (dimensión $n^{[l]} \times 1$)
- $\mathbf{Z}^{[l]}$: Combinación lineal en la capa $l$
- $\mathbf{A}^{[l]}$: Activaciones de la capa $l$

<a name='2.4'></a>
### 2.4 - Forward Propagation

**Forward propagation** calcula las predicciones propagando las entradas hacia adelante a través de todas las capas.

Para una red con 1 capa oculta:

$$\mathbf{Z}^{[1]} = \mathbf{W}^{[1]} \mathbf{X} + \mathbf{b}^{[1]}$$
$$\mathbf{A}^{[1]} = g^{[1]}(\mathbf{Z}^{[1]})$$
$$\mathbf{Z}^{[2]} = \mathbf{W}^{[2]} \mathbf{A}^{[1]} + \mathbf{b}^{[2]}$$
$$\mathbf{A}^{[2]} = g^{[2]}(\mathbf{Z}^{[2]})$$

donde $g^{[l]}$ es la función de activación de la capa $l$.

**Forma matricial** (para $m$ ejemplos):
- $\mathbf{X} \in \mathbb{R}^{m \times n^{[0]}}$: Matriz de entrada ($m$ ejemplos, $n^{[0]}$ features)
- $\mathbf{Z}^{[l]} \in \mathbb{R}^{m \times n^{[l]}}$: Activaciones pre-activación
- $\mathbf{A}^{[l]} \in \mathbb{R}^{m \times n^{[l]}}$: Activaciones post-activación

<a name='2.5'></a>
### 2.5 - Backpropagation y Gradientes

**Backpropagation** es el algoritmo que calcula los gradientes de la función de costo con respecto a todos los parámetros usando la **regla de la cadena**.

#### Función de Costo (Binary Cross-Entropy)

Para clasificación binaria:
$$J(\mathbf{W}, \mathbf{b}) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1-y^{(i)}) \log(1-\hat{y}^{(i)}) \right]$$

#### Gradientes (Red con 1 capa oculta)

**Capa de salida ($l=2$):**
$$d\mathbf{Z}^{[2]} = \mathbf{A}^{[2]} - \mathbf{Y}$$
$$d\mathbf{W}^{[2]} = \frac{1}{m} (\mathbf{A}^{[1]})^T d\mathbf{Z}^{[2]}$$
$$d\mathbf{b}^{[2]} = \frac{1}{m} \sum_{i=1}^{m} d\mathbf{Z}^{[2]}$$

**Capa oculta ($l=1$):**
$$d\mathbf{A}^{[1]} = d\mathbf{Z}^{[2]} (\mathbf{W}^{[2]})^T$$
$$d\mathbf{Z}^{[1]} = d\mathbf{A}^{[1]} \odot g'^{[1]}(\mathbf{Z}^{[1]})$$
$$d\mathbf{W}^{[1]} = \frac{1}{m} \mathbf{X}^T d\mathbf{Z}^{[1]}$$
$$d\mathbf{b}^{[1]} = \frac{1}{m} \sum_{i=1}^{m} d\mathbf{Z}^{[1]}$$

donde $\odot$ denota producto elemento a elemento (Hadamard).

#### Actualización de Parámetros (Gradient Descent)

$$\mathbf{W}^{[l]} := \mathbf{W}^{[l]} - \alpha \, d\mathbf{W}^{[l]}$$
$$\mathbf{b}^{[l]} := \mathbf{b}^{[l]} - \alpha \, d\mathbf{b}^{[l]}$$

donde $\alpha$ es el learning rate.

<a name='3'></a>
## 3 - Implementación desde CERO

<a name='3.1'></a>
### 3.1 - Clase RedNeuronal Completa

Implementaremos una red neuronal con 1 capa oculta usando únicamente NumPy.

In [ ]:
class RedNeuronal:
    """
    Red Neuronal simple (1 capa oculta) desde cero.
    
    Arquitectura:
    - Capa de entrada: n_input features
    - Capa oculta: n_hidden neuronas con activación sigmoid
    - Capa de salida: n_output neuronas con activación sigmoid
    
    Parámetros
    ----------
    input_size : int
        Número de features de entrada
    hidden_size : int
        Número de neuronas en la capa oculta
    output_size : int
        Número de neuronas en la capa de salida (1 para clasificación binaria)
    learning_rate : float, opcional (default=0.01)
        Tasa de aprendizaje para gradient descent
    """
    
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.learning_rate = learning_rate
        
        # Inicializar pesos aleatoriamente (pequeños valores para romper simetría)
        np.random.seed(42)
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        
        self.loss_history = []
    
    def _sigmoid(self, z):
        """Función de activación sigmoide"""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def _sigmoid_derivative(self, a):
        """Derivada de sigmoide (en términos de la activación a)"""
        return a * (1 - a)
    
    def forward(self, X):
        """
        Forward propagation.
        
        Parámetros
        ----------
        X : ndarray
            Matriz de entrada de forma (m, n_input)
        
        Retorna
        -------
        tuple
            (Z1, A1, Z2, A2) - Activaciones de cada capa
        """
        # Capa oculta
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = self._sigmoid(Z1)
        
        # Capa de salida
        Z2 = np.dot(A1, self.W2) + self.b2
        A2 = self._sigmoid(Z2)
        
        return Z1, A1, Z2, A2
    
    def backward(self, X, y, Z1, A1, Z2, A2):
        """
        Backpropagation - calcula gradientes.
        
        Parámetros
        ----------
        X : ndarray
            Matriz de entrada de forma (m, n_input)
        y : ndarray
            Vector de etiquetas verdaderas de forma (m,)
        Z1, A1, Z2, A2 : ndarray
            Activaciones de forward propagation
        
        Retorna
        -------
        tuple
            (dW1, db1, dW2, db2) - Gradientes de todos los parámetros
        """
        m = X.shape[0]
        
        # Gradientes de la capa de salida
        dZ2 = A2 - y.reshape(-1, 1)
        dW2 = (1/m) * np.dot(A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
        
        # Gradientes de la capa oculta
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self._sigmoid_derivative(A1)
        dW1 = (1/m) * np.dot(X.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
        
        return dW1, db1, dW2, db2
    
    def update_parameters(self, dW1, db1, dW2, db2):
        """Actualiza pesos usando gradient descent"""
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
    
    def _binary_cross_entropy(self, y_true, y_pred):
        """Binary Cross-Entropy Loss"""
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X, y, epochs=1000, verbose=True):
        """
        Entrena la red neuronal.
        
        Parámetros
        ----------
        X : ndarray
            Matriz de features de forma (m, n_input)
        y : ndarray
            Vector de etiquetas de forma (m,)
        epochs : int, opcional (default=1000)
            Número de iteraciones de entrenamiento
        verbose : bool, opcional (default=True)
            Si True, imprime progreso cada 100 épocas
        
        Retorna
        -------
        self
        """
        for epoch in range(epochs):
            # Forward propagation
            Z1, A1, Z2, A2 = self.forward(X)
            
            # Calcular loss
            loss = self._binary_cross_entropy(y, A2)
            self.loss_history.append(loss)
            
            # Backpropagation
            dW1, db1, dW2, db2 = self.backward(X, y, Z1, A1, Z2, A2)
            
            # Actualizar parámetros
            self.update_parameters(dW1, db1, dW2, db2)
            
            # Imprimir progreso
            if verbose and (epoch + 1) % 100 == 0:
                print(f"Época {epoch+1}/{epochs}, Loss: {loss:.4f}")
        
        return self
    
    def predict_proba(self, X):
        """Predice probabilidades"""
        _, _, _, A2 = self.forward(X)
        return A2.flatten()
    
    def predict(self, X, threshold=0.5):
        """Predice clases"""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase RedNeuronal implementada correctamente")

<a name='4'></a>
## 4 - Ejercicios GRADED

Ahora es tu turno de implementar los componentes clave de una red neuronal.

<a name='ex01'></a>
### Ejercicio 1 - Forward Propagation

Implementa la función `neural_network_forward` que realiza forward propagation para una red con 1 capa oculta.

**Instrucciones:**
1. Calcula $Z^{[1]} = X W^{[1]} + b^{[1]}$
2. Aplica activación: $A^{[1]} = \sigma(Z^{[1]})$
3. Calcula $Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]}$
4. Aplica activación: $A^{[2]} = \sigma(Z^{[2]})$
5. Retorna todas las activaciones (necesarias para backpropagation)

**Hint:** Usa `np.dot()` para multiplicación matricial y broadcasting para sumar sesgos.

In [ ]:
# GRADED FUNCTION: neural_network_forward

def neural_network_forward(X, W1, b1, W2, b2):
    """
    Realiza forward propagation para una red neuronal con 1 capa oculta.
    
    Parámetros
    ----------
    X : ndarray
        Matriz de entrada de forma (m, n_input)
    W1 : ndarray
        Matriz de pesos de la capa oculta de forma (n_input, n_hidden)
    b1 : ndarray
        Vector de sesgos de la capa oculta de forma (1, n_hidden)
    W2 : ndarray
        Matriz de pesos de la capa de salida de forma (n_hidden, n_output)
    b2 : ndarray
        Vector de sesgos de la capa de salida de forma (1, n_output)
    
    Retorna
    -------
    tuple
        (Z1, A1, Z2, A2) donde:
        - Z1: activaciones pre-activación de la capa oculta (m, n_hidden)
        - A1: activaciones post-activación de la capa oculta (m, n_hidden)
        - Z2: activaciones pre-activación de la capa de salida (m, n_output)
        - A2: activaciones post-activación de la capa de salida (m, n_output)
    """
    
    def sigmoid(z):
        """Función auxiliar sigmoide"""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    ### YOUR CODE STARTS HERE ###
    # Capa oculta
    Z1 = None  # Calcula Z1 = X @ W1 + b1
    A1 = None  # Aplica sigmoid a Z1
    
    # Capa de salida
    Z2 = None  # Calcula Z2 = A1 @ W2 + b2
    A2 = None  # Aplica sigmoid a Z2
    ### YOUR CODE ENDS HERE ###
    
    return Z1, A1, Z2, A2

In [ ]:
# Test para Ejercicio 1
from tests.deep_learning.test_01_redes_neuronales import test_ejercicio_1_forward

verificar_forward = test_ejercicio_1_forward()
verificar_forward(neural_network_forward)

**Salida esperada:**
```
✅ Test 1 aprobado: Dimensiones correctas
✅ Test 2 aprobado: Valores de Z1 correctos
✅ Test 3 aprobado: Valores de A1 correctos (sigmoid aplicado)
✅ Test 4 aprobado: Valores de Z2 correctos
✅ Test 5 aprobado: Valores de A2 correctos (probabilidades entre 0 y 1)
✅ Test 6 aprobado: Forward propagation completa funciona correctamente

🎉 ¡Todos los tests pasaron! Ejercicio 1 completado exitosamente.
```

<a name='ex02'></a>
### Ejercicio 2 - Calcular Gradientes (Backpropagation)

Implementa la función `compute_gradients` que calcula los gradientes para la capa oculta usando backpropagation.

**Instrucciones:**
1. Calcula $dZ^{[2]} = A^{[2]} - Y$ (gradiente de la capa de salida)
2. Propaga hacia atrás: $dA^{[1]} = dZ^{[2]} (W^{[2]})^T$
3. Calcula $dZ^{[1]} = dA^{[1]} \odot g'(Z^{[1]})$ donde $g'$ es la derivada de sigmoid
4. Calcula los gradientes: $dW^{[1]} = \frac{1}{m} X^T dZ^{[1]}$
5. Calcula el gradiente del sesgo: $db^{[1]} = \frac{1}{m} \sum dZ^{[1]}$

**Hint:** Para sigmoid, $g'(z) = g(z)(1 - g(z))$. Usa `*` para producto elemento a elemento.

In [ ]:
# GRADED FUNCTION: compute_gradients

def compute_gradients(X, y, A1, A2, W2):
    """
    Calcula los gradientes de la capa oculta usando backpropagation.
    
    Parámetros
    ----------
    X : ndarray
        Matriz de entrada de forma (m, n_input)
    y : ndarray
        Vector de etiquetas verdaderas de forma (m,) o (m, 1)
    A1 : ndarray
        Activaciones de la capa oculta de forma (m, n_hidden)
    A2 : ndarray
        Activaciones de la capa de salida de forma (m, 1)
    W2 : ndarray
        Matriz de pesos de la capa de salida de forma (n_hidden, 1)
    
    Retorna
    -------
    tuple
        (dW1, db1) donde:
        - dW1: gradiente de W1 de forma (n_input, n_hidden)
        - db1: gradiente de b1 de forma (1, n_hidden)
    """
    
    m = X.shape[0]
    y = y.reshape(-1, 1)  # Asegurar forma correcta
    
    ### YOUR CODE STARTS HERE ###
    # Gradiente de la capa de salida
    dZ2 = None  # A2 - y
    
    # Propagar hacia atrás a la capa oculta
    dA1 = None  # dZ2 @ W2.T
    
    # Aplicar derivada de sigmoid: g'(z) = a * (1 - a)
    dZ1 = None  # dA1 * (A1 * (1 - A1))
    
    # Calcular gradientes de W1 y b1
    dW1 = None  # (1/m) * X.T @ dZ1
    db1 = None  # (1/m) * np.sum(dZ1, axis=0, keepdims=True)
    ### YOUR CODE ENDS HERE ###
    
    return dW1, db1

In [ ]:
# Test para Ejercicio 2
from tests.deep_learning.test_01_redes_neuronales import test_ejercicio_2_gradients

verificar_gradients = test_ejercicio_2_gradients()
verificar_gradients(compute_gradients)

**Salida esperada:**
```
✅ Test 1 aprobado: Dimensiones de dW1 correctas
✅ Test 2 aprobado: Dimensiones de db1 correctas
✅ Test 3 aprobado: Valores de dW1 correctos
✅ Test 4 aprobado: Valores de db1 correctos
✅ Test 5 aprobado: Gradientes calculados correctamente para diferentes datos

🎉 ¡Todos los tests pasaron! Ejercicio 2 completado exitosamente.
```

<a name='5'></a>
## 5 - Ejemplo Práctico: Clasificación No Lineal

<a name='5.1'></a>
### 5.1 - Dataset Moons

Usaremos el dataset **Moons** de sklearn, que tiene una forma no lineal (dos medias lunas entrelazadas). Este dataset es ideal para demostrar el poder de las redes neuronales.

In [ ]:
from sklearn.datasets import make_moons

# Generar datos no lineales (forma de luna)
X, y = make_moons(n_samples=300, noise=0.2, random_state=42)

print(f"📊 Dataset generado:")
print(f"   - Número de ejemplos: {X.shape[0]}")
print(f"   - Número de features: {X.shape[1]}")
print(f"   - Distribución de clases: Clase 0: {np.sum(y==0)}, Clase 1: {np.sum(y==1)}")

# Visualizar
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k', s=50)
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k', s=50)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Dataset Moons - Datos No Lineales', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 Observa que las clases NO son linealmente separables.")
print("   Una línea recta no puede separar perfectamente las dos medias lunas.")

<a name='5.2'></a>
### 5.2 - Entrenamiento de la Red

Entrenemos una red neuronal con:
- **Entrada**: 2 features
- **Capa oculta**: 4 neuronas
- **Salida**: 1 neurona (clasificación binaria)

In [ ]:
# Crear y entrenar red neuronal
# input_size=2 (2 features), hidden_size=4 (4 neuronas ocultas), output_size=1
nn = RedNeuronal(input_size=2, hidden_size=4, output_size=1, learning_rate=0.5)

print("🚀 Iniciando entrenamiento...\n")
nn.fit(X, y, epochs=2000, verbose=True)

# Evaluar
accuracy = nn.score(X, y)
print(f"\n✅ Accuracy final: {accuracy:.4f} ({accuracy*100:.2f}%)")

<a name='5.3'></a>
### 5.3 - Visualización de Resultados

In [ ]:
# Visualizar frontera de decisión
plot_decision_boundary(X, y, nn, title="Red Neuronal - Frontera de Decisión No Lineal")
plt.show()

print("\n💡 La red neuronal ha aprendido una frontera de decisión NO LINEAL")
print("   que separa correctamente las dos medias lunas.")

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(10, 6))
plt.plot(nn.loss_history, linewidth=2, color='#2E86AB')
plt.xlabel('Época', fontsize=12)
plt.ylabel('Binary Cross-Entropy Loss', fontsize=12)
plt.title('Curva de Aprendizaje - Red Neuronal', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f"📉 Loss inicial: {nn.loss_history[0]:.4f}")
print(f"📉 Loss final: {nn.loss_history[-1]:.4f}")
print(f"📉 Reducción: {(1 - nn.loss_history[-1]/nn.loss_history[0])*100:.2f}%")

<a name='5.4'></a>
### 5.4 - Comparación con Regresión Logística

Comparemos el desempeño de la red neuronal con un modelo lineal simple.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Entrenar regresión logística
lr = LogisticRegression()
lr.fit(X, y)

print(f"Accuracy Regresión Logística: {lr.score(X, y):.4f}")
print(f"Accuracy Red Neuronal: {nn.score(X, y):.4f}")
print(f"\n✨ Mejora: {(nn.score(X, y) - lr.score(X, y))*100:.2f} puntos porcentuales")

In [ ]:
# Comparar fronteras de decisión
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Crear mesh para visualización
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Regresión Logística
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()])
Z_lr = Z_lr.reshape(xx.shape)

axes[0].contourf(xx, yy, Z_lr, alpha=0.3, cmap='RdBu', levels=1)
axes[0].scatter(X[y==0, 0], X[y==0, 1], c='red', edgecolors='k', s=50)
axes[0].scatter(X[y==1, 0], X[y==1, 1], c='blue', edgecolors='k', s=50)
axes[0].set_title(f'Regresión Logística (Lineal)\nAccuracy: {lr.score(X, y):.4f}', 
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# Red Neuronal
Z_nn = nn.predict(np.c_[xx.ravel(), yy.ravel()])
Z_nn = Z_nn.reshape(xx.shape)

axes[1].contourf(xx, yy, Z_nn, alpha=0.3, cmap='RdBu', levels=1)
axes[1].scatter(X[y==0, 0], X[y==0, 1], c='red', edgecolors='k', s=50)
axes[1].scatter(X[y==1, 0], X[y==1, 1], c='blue', edgecolors='k', s=50)
axes[1].set_title(f'Red Neuronal (No Lineal)\nAccuracy: {nn.score(X, y):.4f}', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Observaciones:")
print("   - Regresión Logística: Frontera LINEAL (línea recta)")
print("   - Red Neuronal: Frontera NO LINEAL (curva adaptativa)")
print("   ✅ La Red Neuronal puede aprender patrones más complejos!")

### 🎯 Ejercicio Adicional: Experimentar con Arquitectura

Prueba diferentes configuraciones y observa cómo afectan el desempeño.

In [ ]:
# TU CÓDIGO AQUÍ
# Experimenta con:
# 1. Diferente número de neuronas ocultas (2, 8, 16, 32)
# 2. Diferentes learning rates (0.1, 1.0, 2.0)
# 3. Diferentes épocas (500, 1000, 5000)
# 4. Visualiza cómo cambia la frontera de decisión

# Ejemplo:
# nn_experiment = RedNeuronal(input_size=2, hidden_size=8, output_size=1, learning_rate=1.0)
# nn_experiment.fit(X, y, epochs=1000, verbose=False)
# print(f"Accuracy: {nn_experiment.score(X, y):.4f}")

print("💡 Experimenta con diferentes configuraciones aquí")

<a name='6'></a>
## 6 - Resumen y Conceptos Clave

### ✅ Conceptos Fundamentales

1. **Neurona Artificial**: Unidad básica que realiza combinación lineal + activación no lineal
2. **Forward Propagation**: Propagación de datos de entrada a salida a través de las capas
3. **Backpropagation**: Cálculo eficiente de gradientes usando la regla de la cadena
4. **Funciones de Activación**: Introducen no linealidad (sigmoid, tanh, ReLU)
5. **Gradient Descent**: Algoritmo de optimización para actualizar pesos
6. **Poder Expresivo**: Capas ocultas permiten aprender patrones no lineales complejos

### 🎯 Lecciones Aprendidas

- ✅ Las redes neuronales pueden **aproximar cualquier función** (Teorema de Aproximación Universal)
- ✅ Más neuronas ocultas → Mayor capacidad, pero riesgo de **overfitting**
- ✅ Learning rate muy alto → Divergencia; muy bajo → Convergencia lenta
- ✅ La inicialización de pesos es crítica para romper simetría
- ✅ Backpropagation hace factible el entrenamiento de redes profundas

### 🚀 Conceptos Avanzados (Deep Learning)

- **Redes Profundas**: Múltiples capas ocultas (Deep Neural Networks)
- **Arquitecturas Especializadas**: 
  - CNN (Convolutional Neural Networks) para imágenes
  - RNN/LSTM para secuencias y series temporales
  - Transformers para NLP
- **Regularización**: Dropout, L2, Batch Normalization
- **Optimizadores Avanzados**: Adam, RMSprop, AdaGrad
- **Transfer Learning**: Reutilizar modelos pre-entrenados

<div style="background-color: #E3F2FD; padding: 20px; border-left: 5px solid #2196F3; border-radius: 5px; margin: 20px 0;">
    <h3 style="color: #1976D2; margin-top: 0;">💡 ¿Por qué Redes Neuronales?</h3>
    <p style="margin-bottom: 10px;"><strong>Comparación con otros algoritmos:</strong></p>
    <ul style="margin-bottom: 10px;">
        <li><strong>Regresión Logística</strong>: Solo fronteras lineales. Simple pero limitado.</li>
        <li><strong>K-NN</strong>: No lineal pero costoso en predicción y sensible a ruido.</li>
        <li><strong>Árboles de Decisión</strong>: No lineales pero fronteras "rectangulares".</li>
        <li><strong>Redes Neuronales</strong>: Fronteras arbitrariamente complejas, escalables, universales.</li>
    </ul>
    <p style="margin-bottom: 0;"><strong>Casos de uso ideales:</strong></p>
    <ul style="margin-bottom: 0;">
        <li>✅ Visión por computadora (reconocimiento de imágenes)</li>
        <li>✅ Procesamiento de lenguaje natural (traducción, chatbots)</li>
        <li>✅ Reconocimiento de voz y generación de audio</li>
        <li>✅ Sistemas de recomendación personalizados</li>
        <li>✅ Juegos y robótica (Reinforcement Learning)</li>
    </ul>
</div>

### 🎓 ¡Felicidades!

Has implementado una **Red Neuronal desde CERO** y comprendido los fundamentos del Deep Learning. Estos conceptos son la base de modelos modernos como GPT, BERT, ResNet, y muchos más.

<a name='7'></a>
## 7 - Referencias

### 📚 Papers Fundamentales

1. **McCulloch, W. S., & Pitts, W. (1943)**. "A logical calculus of the ideas immanent in nervous activity." *Bulletin of Mathematical Biophysics*, 5(4), 115-133.
   - Paper original sobre neuronas artificiales

2. **Rosenblatt, F. (1958)**. "The perceptron: A probabilistic model for information storage and organization in the brain." *Psychological Review*, 65(6), 386.
   - Introducción del Perceptrón

3. **Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986)**. "Learning representations by back-propagating errors." *Nature*, 323(6088), 533-536.
   - Popularización de Backpropagation

4. **Hornik, K., Stinchcombe, M., & White, H. (1989)**. "Multilayer feedforward networks are universal approximators." *Neural Networks*, 2(5), 359-366.
   - Teorema de Aproximación Universal

5. **LeCun, Y., Bengio, Y., & Hinton, G. (2015)**. "Deep learning." *Nature*, 521(7553), 436-444.
   - Review moderno de Deep Learning

### 📖 Libros Recomendados

1. **Goodfellow, I., Bengio, Y., & Courville, A. (2016)**. *Deep Learning*. MIT Press.
   - Biblia del Deep Learning, disponible gratis: https://www.deeplearningbook.org/

2. **Nielsen, M. A. (2015)**. *Neural Networks and Deep Learning*. Determination Press.
   - Introducción visual e intuitiva: http://neuralnetworksanddeeplearning.com/

3. **Bishop, C. M. (2006)**. *Pattern Recognition and Machine Learning*. Springer.
   - Enfoque matemático riguroso

### 🌐 Recursos Online

1. **CS231n - Stanford**: Convolutional Neural Networks for Visual Recognition
   - http://cs231n.stanford.edu/

2. **3Blue1Brown - Neural Networks Series**: Visualizaciones excelentes
   - https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi

3. **Distill.pub**: Artículos interactivos sobre ML/DL
   - https://distill.pub/

4. **TensorFlow Playground**: Experimenta con redes neuronales en el navegador
   - https://playground.tensorflow.org/

### 💻 Frameworks de Deep Learning

1. **PyTorch**: https://pytorch.org/ - Framework más popular en investigación
2. **TensorFlow/Keras**: https://www.tensorflow.org/ - Framework de Google
3. **JAX**: https://github.com/google/jax - NumPy acelerado con GPU
4. **FastAI**: https://www.fast.ai/ - Abstracción de alto nivel sobre PyTorch